# 1. Data selection, collection and cleaning

**Methodology step 2 (MSR): selection and collection of data.**

* **Source:** GitSkills, July 2026 snapshot (Zenodo 10.5281/zenodo.21875637): every `SKILL.md` file found by GitHub code search, with repository metadata, bundled files and sampled commit history.
* **Units of analysis:** an *occurrence* is one file at one repository path (used for adoption and reuse); a *skill* is one distinct content, identified by its git blob hash (used for text and security analyses).
* **Selection criteria:** the *strict* population keeps skills with the exact `SKILL.md` basename, valid front matter, a verified content hash, no symlink, and a first commit on or after October 2025, when the format was released.
* **Quality validation:** we report how many skills each filter removes and how complete the commit history is per location class.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
print("database:", os.environ["GITSKILLS_DB"])

## 1.1 Dataset size

In [ ]:
counts = db.table_counts()
counts

## 1.2 Load the processed tables
`scripts/build_features.py` streams the database once and writes one row per skill (with features) and one row per occurrence.

In [ ]:
skills = clean.load_skills()
occ = clean.load_occurrences()
print(f"{len(skills):,} distinct skills, {len(occ):,} occurrences, {occ.repo_full_name.nunique():,} repositories")

## 1.3 Selection criteria: the strict population
Each row applies one more filter (cumulative).

In [ ]:
pop = clean.population_report(skills)
save_table(pop, "eda_population_filters")
pop

## 1.4 Where skill files live
`canonical` = `.claude/skills/<name>/SKILL.md`; `skills-dir` = any other `skills/` folder; `other` = anywhere else.

In [ ]:
occ["location_class"].value_counts(normalize=True).rename("share").to_frame()

In [ ]:
occ["filename"].value_counts().head(10).rename("files").to_frame()

## 1.5 Quality: commit-history coverage
Dates exist only for a sample. Time series use only location classes with near-complete coverage.

In [ ]:
cov = adoption.history_coverage(clean.strict(skills))
save_table(cov, "rq1_history_coverage")
cov

## 1.6 Exploratory view: copies per skill and body length

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 2.6))
vc = skills["copies"].value_counts().sort_index()
axes[0].loglog(vc.index, vc.values, ".", ms=3)
axes[0].set(xlabel="copies of a distinct skill", ylabel="# skills")
b = skills["body_chars"].dropna(); b = b[b > 0]
axes[1].hist(np.log10(b), bins=50)
axes[1].set(xlabel="log10(body characters)", ylabel="# skills")
plt.tight_layout(); plt.show()